<a href="https://colab.research.google.com/github/denmex/nyc_payroll_hw/blob/main/NYC_Payroll_HW2_ETL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
print("NYC PAYROLL ETL PIPELINE")

NYC PAYROLL ETL PIPELINE


In [ ]:
from google.colab import files
import pandas as pd
import uuid
uploaded = files.upload()

Saving Citywide_Payroll_Data_(Fiscal_Year)_20260518.csv to Citywide_Payroll_Data_(Fiscal_Year)_20260518 (3).csv


In [ ]:
df = pd.read_csv("Citywide_Payroll_Data_(Fiscal_Year)_20260518.csv", low_memory=False)

print("Loaded:", df.shape)

Loaded: (550219, 17)


In [ ]:
df = df.sample(n=50000, random_state=42)

print("Sampled:", df.shape)

Sampled: (50000, 17)


In [ ]:
numeric_cols = [
    "Base Salary",
    "Regular Hours",
    "Regular Gross Paid",
    "OT Hours",
    "Total OT Paid",
    "Total Other Pay"
]

In [ ]:
for col in numeric_cols:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace(",", "")
        .str.replace("$", "")
    )
    df[col] = pd.to_numeric(df[col], errors="coerce")

df[numeric_cols] = df[numeric_cols].fillna(0)

In [ ]:
def anonymize_person(first, middle, last):
    full_name = (
        str(first if pd.notnull(first) else "").strip().lower() +
        str(middle if pd.notnull(middle) else "").strip().lower() +
        str(last if pd.notnull(last) else "").strip().lower()
    )
    return str(uuid.uuid5(uuid.NAMESPACE_DNS, full_name))

In [ ]:
df["employee_id"] = df.apply(
    lambda row: anonymize_person(
        row["First Name"],
        row["Mid Init"],
        row["Last Name"]
    ),
    axis=1
)

In [ ]:
df = df.drop(columns=["First Name", "Mid Init", "Last Name"])

In [ ]:
df_clean = df[
    (df["Base Salary"] > 0) &
    (df["Regular Gross Paid"] > 0)
].copy()


print("Clean dataset:", df_clean.shape)

Clean dataset: (48481, 15)


In [ ]:
#star schema

In [ ]:
fact_payroll = df_clean[[
    "employee_id",
    "Fiscal Year",
    "Payroll Number",
    "Agency Name",
    "Title Description",
    "Base Salary",
    "Regular Hours",
    "Regular Gross Paid",
    "OT Hours",
    "Total OT Paid",
    "Total Other Pay"
]].copy()

print("FACT:", fact_payroll.shape)

FACT: (48481, 11)


In [ ]:
dim_employee = df_clean[[
    "employee_id",
    "Work Location Borough"
]].drop_duplicates()

In [ ]:
dim_department = df_clean[[
    "Payroll Number",
    "Agency Name"
]].drop_duplicates()

In [ ]:
dim_role = df_clean[[
    "Title Description",
    "Pay Basis"
]].drop_duplicates()

In [ ]:
dim_time = df_clean[[
    "Fiscal Year"
]].drop_duplicates()

In [ ]:
print("NULL CHECK:")
print(df_clean.isnull().sum())

print("FACT SAMPLE:")
print(fact_payroll.head())

NULL CHECK:
Fiscal Year                   0
Payroll Number                0
Agency Name                   0
Agency Start Date             0
Work Location Borough         0
Title Description             0
Leave Status as of June 30    0
Base Salary                   0
Pay Basis                     0
Regular Hours                 0
Regular Gross Paid            0
OT Hours                      0
Total OT Paid                 0
Total Other Pay               0
employee_id                   0
dtype: int64
FACT SAMPLE:
                                 employee_id  Fiscal Year  Payroll Number  \
354868  442419c9-ff75-5b33-aa8f-4383e661a4c3         2025             747   
504215  7b3746f7-1860-52a9-ac45-715521993846         2025              56   
218661  a7ee6f1e-2b38-5fc8-8477-453429eaf05b         2025             742   
285401  9b181dd8-af1a-53bd-b810-d92af88eaee6         2025             742   
373386  230b7753-cd05-5dec-80ab-f22e5277279c         2025             747   

                   

In [ ]:
df_clean.to_csv("clean_payroll.csv", index=False)

files.download("clean_payroll.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>